<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# ⚖️ AI Debate Arena: GRPO vs Distillation

Two agents argue opposite sides of an open research question, each citing papers
from the index built in notebook 02. A third weighs the arguments, declares a
winner, and implements it.

**The question**: to improve reasoning in a small (under-10B) model on tasks with
programmatically verifiable answers — given both a reward checker and a large
reasoning teacher — should you post-train with **GRPO** (reinforcement learning
on verifiable rewards) or **distil the teacher's reasoning traces** (SFT)?

## Why this question

A debate is only worth staging if the answer is genuinely unsettled. This one is:
the 2025–2026 literature contains head-to-head comparisons pointing both ways,
and results that hold for a 70B model often reverse for a 7B one.

It is also framed so that **either verdict leads somewhere runnable on this
platform** — GRPO to the `zebra-grpo` notebook, distillation to collecting traces
from your served model and fine-tuning a student. A debate whose losing outcome
is a dead end is a demonstration, not a decision.

## The pattern

```
                    ROUND 1: Opening arguments (parallel)
                    ┌────────────────┬────────────────┐
                    ▼                ▼
            🔵 Pro-GRPO       🔴 Pro-Distillation
            searches papers    searches papers
            presents case      presents case
                    └───────┬────────┘
                            ▼
                    ROUND 2: Rebuttals (parallel)
                    ┌────────────────┬────────────────┐
                    ▼                ▼
            🔵 Pro-GRPO       🔴 Pro-Distillation
            reads opponent     reads opponent
            counters it        counters it
                    └───────┬────────┘
                            ▼
                    ⚖️ Judge — five criteria, equally weighted
                       declares a winner, calls generate_code
                            ▼
                    💻 An implementation of whichever side won
```

Rebuttals are what make it a debate rather than two monologues: each side reads
the other's argument and answers it, which surfaces the strongest objection to
each position rather than the strongest case for it.

## The agents

| Agent | Argues for | Tool |
|---|---|---|
| **Pro_GRPO** | RL on verifiable rewards | `search_papers` |
| **Pro_Distill** | Distilling a reasoning teacher | `search_papers` |
| **Judge** | Neither — scores both, then implements the winner | `generate_code` |

## How the judge decides

Five criteria, **equally weighted at 20% each**: quality, efficiency,
simplicity, practicality, robustness.

Equal weighting is deliberate. Benchmark quality is the number papers compete on,
so weighting it heavily would let the debate be won by whoever cites the most
favourable table. For someone choosing a method to actually run, whether it fits
on one GPU and how many hyperparameters it needs matter as much as a point of
accuracy.

**Prerequisites**: run `02-langchain-rag.ipynb` first — this notebook reads the
Qdrant collection it builds. A full debate takes about four minutes.

---
## Setup

**What**: Connect to the LLM Gateway, Qdrant and Langfuse, and pick the models.

**Why the model choice matters here**: this notebook depends on **tool calling**.
The agents cannot cite papers unless the model can emit a well-formed call to
`search_papers`, so the selection below prefers a model whose serving stack
advertises tool support, rather than simply taking the first one loaded.

**How**: `tk-llm` discovers the gateway; everything else comes from environment
variables that JupyterHub injects from service discovery.

In [18]:
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown, HTML
from tk_llm import LLMClient, get_openai_client

# Helper functions for colored output
def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✅ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ️  {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">❌ {msg}</span>'))

# Initialize tk-llm
llm_mgmt = LLMClient()
oai_client = get_openai_client()

# Discover loaded models
# This notebook uses AG2 tool calling, so we need a model with native tool support.
# The Ollama library models (source=ollama) include proper RENDERER/PARSER for tools,
# unlike unsloth GGUFs which are text-only and lack tool calling templates.
available = llm_mgmt.list_models(state="available")

# Prefer Ollama library models (id starts with "ollama/") for tool support
CHAT_MODEL = (
    next((m.id for m in available.models if m.task == "text-generation" and m.id.startswith("ollama/")), None)
    or next((m.id for m in available.models if m.task == "text-generation" and m.tool_use), None)
    or next((m.id for m in available.models if m.task == "text-generation"), None)
)
EMBED_MODEL = next((m.id for m in available.models if m.task == "feature-extraction"), None)

# Get gateway config for AG2/AutoGen
LLM_GATEWAY_URL = os.environ.get('LLM_GATEWAY_URL', 'https://llm.' + os.environ.get('DOMAIN_NAME', 'thinkube.com'))
LLM_GATEWAY_TOKEN = os.environ.get('THINKUBE_API_TOKEN', 'not-needed')

# Other platform services
QDRANT_URL = os.environ.get('QDRANT_URL')
LANGFUSE_HOST = os.environ.get('LANGFUSE_HOST')
LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY')

info(f"Chat model: {CHAT_MODEL}")
info(f"Embedding model: {EMBED_MODEL}")
info(f"Qdrant: {QDRANT_URL}")
info(f"Langfuse: {LANGFUSE_HOST}")

---
## 1. Connect to Qdrant

**What**: Open the collection notebook 02 built, and confirm what is in it.

**Why**: everything downstream is only as good as this index. The counts printed
here are worth a glance before starting a four-minute debate — a collection with
no `setup` or `discussion` chunks means notebook 02 ran with `FETCH_FULL_TEXT`
off, and the debaters will be arguing from abstracts alone.

In [ ]:
from qdrant_client import QdrantClient

# Connect to Qdrant
qdrant = QdrantClient(url=QDRANT_URL, port=443, https=True, verify=False)
COLLECTION_NAME = "rl_reasoning_papers"

# Check collection
try:
    collection_info = qdrant.get_collection(COLLECTION_NAME)
    success(f"Connected to Qdrant collection '{COLLECTION_NAME}'")
    info(f"Total vectors: {collection_info.points_count}")
except Exception as e:
    error(f"Collection not found. Run notebook 02 first!")
    raise e

# Get unique papers from the collection
papers_result = qdrant.scroll(collection_name=COLLECTION_NAME, limit=500, with_payload=True)
unique_papers = {}
for point in papers_result[0]:
    meta = point.payload.get('metadata', {})
    pid = meta.get('paper_id')
    if pid and pid not in unique_papers:
        unique_papers[pid] = {
            'title': meta.get('title', 'Unknown'),
            'authors': meta.get('authors', 'Unknown'),
            'published_date': meta.get('published_date', 'Unknown')
        }

success(f"Found {len(unique_papers)} unique papers")

# Display sample papers
print("\nSample papers available:")
for i, (pid, paper) in enumerate(list(unique_papers.items())[:5], 1):
    print(f"  {i}. {paper['title'][:70]}...")

---
## 2. Define the Tools

Agents cannot browse. Everything they know about the literature arrives through
two functions, so how those functions retrieve *is* the quality of the debate.

### `search_papers` — used by both advocates

Embeds the query and returns five chunks. The interesting part is which five.

Plain top-5 similarity was tried first and failed in two distinct ways, both
measured on this corpus:

1. **The most numerous content type takes every slot.** Ask about mechanism, get
   benchmark tables.
2. **Foundational papers never appear.** A paper that *introduced* a method
   mentions it a handful of times; fifty later papers discuss it on every page
   and win on similarity. In one run the debaters never once cited the paper
   that introduced GRPO, though it sat in the index.

So the slots are reserved:

| slots | drawn from | role |
|---|---|---|
| 3 | `abstract`, `setup`, `discussion` | what is claimed, under what conditions, and where it breaks |
| 1 | `results`, `algorithm`, `equation` | the numbers and the formal statement |
| 1 | any paper marked `anchor` | the founding work, and the work that tested it and disagreed |

An explicit `content_type` argument bypasses all of this and searches that type
alone — useful when an agent specifically wants limitations or tables.

### `generate_code` — used by the judge

Retrieves algorithm chunks for the **winning** approach and writes an
implementation grounded in them. The two sides need genuinely different code — a
GRPO training loop with group-relative advantages, or a rejection-sampling and
SFT pipeline over teacher traces — so the tool carries a separate requirement
list per side rather than one generic prompt.

Note that it disables thinking on its own LLM call: it is a pure code request,
and a reasoning model that spends its budget deliberating returns nothing.

In [ ]:
from typing import Annotated
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny

# Store extracted context for transparency
extracted_contexts = []

# Chunks divide into what a paper argues and what it measures. Retrieval draws
# from both so an argument cannot be built from benchmark grids alone.
PROSE_TYPES = ["abstract", "setup", "discussion"]
EVIDENCE_TYPES = ["results", "algorithm", "equation"]


def _search_by_type(query_vector, types, limit):
    """Nearest chunks restricted to the given content types."""
    return qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(must=[FieldCondition(
            key="content_type", match=MatchAny(any=types))]),
        limit=limit,
        with_payload=True,
    ).points


def _search_anchors(query_vector, limit):
    """Nearest chunks from the anchor papers.

    Anchors are the works that define the debate or that tested it and
    disagreed. They are few, and each is discussed at length by many later
    papers, so on raw similarity they never reach the top five. Reserving a
    slot is what puts the founding paper and its strongest critic in front of
    the debaters.
    """
    return qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(must=[FieldCondition(
            key="category", match=MatchValue(value="anchor"))]),
        limit=limit,
        with_payload=True,
    ).points

def search_papers(
    query: Annotated[str, "Search query for finding relevant papers"],
    content_type: Annotated[str, "Type of content to search: 'abstract', 'algorithm', 'results', 'setup', 'discussion', or 'all'"] = "all"
) -> str:
    """
    Search the paper database using semantic similarity.
    
    Args:
        query: Search query for finding relevant papers
        content_type: Filter by content type:
            - 'abstract': Paper abstracts and summaries
            - 'algorithm': Algorithm pseudocode and implementations
            - 'results': Results tables with performance metrics
            - 'setup': Experimental-setup prose (model sizes, rewards, budgets)
            - 'discussion': Limitations / discussion / conclusion prose
            - 'all': All content types (default)
    
    Returns top 5 most relevant papers with titles, authors, and excerpts.
    Used by BOTH advocates to find evidence for their positions.
    """
    print(f"\n  [TOOL] search_papers: '{query[:50]}...' (type: {content_type})")
    
    # Embed the query using the platform's embedding model
    response = oai_client.embeddings.create(model=EMBED_MODEL, input=query)
    query_vector = response.data[0].embedding
    
    if content_type and content_type != "all":
        # An explicit request is answered literally: the caller asked for
        # tables, or for limitations, and gets only those.
        results = _search_by_type(query_vector, [content_type], 5)
    else:
        # Plain top-5 lets one kind of chunk win every slot. Two effects were
        # measured: derivative papers, which repeat a method's name on every
        # page, crowd out the paper that introduced it; and whichever content
        # type is most numerous crowds out the rest. So the slots are reserved.
        #
        #   prose    — abstracts, experimental setup, limitations: the claims
        #              and the conditions they were measured under
        #   evidence — tables, algorithms, equations: the numbers behind them
        #   anchor   — the definitional and counter-evidence papers, which lose
        #              on similarity to any paper that discusses them at length
        results = (
            _search_by_type(query_vector, PROSE_TYPES, 3)
            + _search_by_type(query_vector, EVIDENCE_TYPES, 1)
            + _search_anchors(query_vector, 1)
        )
        # An anchor is also indexed by content type, so it can arrive twice.
        seen, deduped = set(), []
        for hit in results:
            if hit.id not in seen:
                seen.add(hit.id)
                deduped.append(hit)
        results = deduped
    
    if not results:
        return json.dumps({"error": f"No papers found for: {query}"})
    
    papers = []
    for hit in results:
        meta = hit.payload.get('metadata', {})
        papers.append({
            "paper_id": meta.get('paper_id', 'unknown'),
            "title": meta.get('title', 'Unknown'),
            "authors": meta.get('authors', 'Unknown')[:60],
            "content_type": hit.payload.get('content_type', 'abstract'),
            "excerpt": hit.payload.get('text', '')[:600],
            "relevance_score": round(hit.score, 3)
        })
    
    print(f"  [TOOL] Found {len(papers)} papers (types: {set(p['content_type'] for p in papers)})")
    return json.dumps(papers, indent=2)


def generate_code(
    description: Annotated[str, "What code to generate (e.g., 'GRPO training loop for math reasoning')"],
    winning_approach: Annotated[str, "The winning approach from the debate: 'GRPO' or 'Distillation'"]
) -> str:
    """
    Generate Python code based on ACTUAL paper algorithms from Qdrant.
    
    This tool:
    1. Searches Qdrant for algorithm content related to the winning approach
    2. Extracts algorithm descriptions and pseudocode from papers
    3. Uses those as grounding for code generation
    
    Used by the JUDGE to implement the winning approach.
    """
    print(f"\n  [TOOL] generate_code: '{description[:50]}...'")
    print(f"  [TOOL] Searching for algorithm content for: {winning_approach}")
    
    # The two debate sides need different code: a GRPO training loop, or a
    # distillation (teacher traces -> SFT) pipeline. Everything specific to
    # one side lives in this table.
    approach_specs = {
        "grpo": {
            "label": "GRPO (reinforcement learning on verifiable rewards)",
            "queries": [
                "GRPO group relative policy optimization training algorithm",
                "reinforcement learning verifiable reward policy update implementation",
            ],
            "requirements": """Generate a complete, well-commented PyTorch implementation that:
1. Defines a verifiable reward function (checks a model's final answer against a ground-truth answer, returns 1.0 or 0.0)
2. Samples a GROUP of completions per prompt and computes group-relative advantages (reward minus group mean, divided by group std)
3. Implements one GRPO update step over the policy log-probabilities with a clipped surrogate objective and a KL penalty against a frozen reference model
4. Wraps the policy with LoRA adapters so the loop fits on a single 24 GB GPU
5. Includes a configuration dataclass (group size, KL coefficient, clip range, learning rate)
6. Cites relevant paper concepts in comments where applicable""",
        },
        "distillation": {
            "label": "reasoning distillation (SFT on a teacher's traces)",
            "queries": [
                "reasoning distillation teacher traces supervised fine-tuning algorithm",
                "chain-of-thought distillation rejection sampling implementation",
            ],
            "requirements": """Generate a complete, well-commented Python implementation that:
1. Collects reasoning traces from a large teacher model through an OpenAI-compatible chat API
2. Applies rejection sampling: keeps only traces whose final answer a verifier function confirms as correct
3. Builds an SFT dataset of (prompt, reasoning trace + answer) pairs in chat format
4. Fine-tunes a small student model on that dataset with LoRA adapters (TRL SFTTrainer-style loop) so it fits on a single 24 GB GPU
5. Includes a configuration dataclass (teacher model id, samples per prompt, max trace length, learning rate)
6. Cites relevant paper concepts in comments where applicable""",
        },
    }
    spec_key = "grpo" if "grpo" in winning_approach.lower() or "rl" in winning_approach.lower() else "distillation"
    spec = approach_specs[spec_key]
    search_queries = spec["queries"]
    
    algorithm_filter = Filter(
        must=[FieldCondition(key="content_type", match=MatchValue(value="algorithm"))]
    )
    
    algo_results = []
    seen_ids = set()
    for q in search_queries:
        response = oai_client.embeddings.create(model=EMBED_MODEL, input=q)
        query_vector = response.data[0].embedding
        
        hits = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            query_filter=algorithm_filter,
            limit=3,
            with_payload=True
        ).points
        
        for hit in hits:
            pid = hit.payload.get('metadata', {}).get('paper_id', '')
            if pid not in seen_ids:
                seen_ids.add(pid)
                algo_results.append(hit)
    
    # Also search abstracts for broader context
    abstract_filter = Filter(
        must=[FieldCondition(key="content_type", match=MatchValue(value="abstract"))]
    )
    abstract_query = f"{winning_approach} reasoning post-training language model"
    response = oai_client.embeddings.create(model=EMBED_MODEL, input=abstract_query)
    query_vector = response.data[0].embedding
    
    abstract_results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=abstract_filter,
        limit=3,
        with_payload=True
    ).points
    
    for hit in abstract_results:
        pid = hit.payload.get('metadata', {}).get('paper_id', '')
        if pid not in seen_ids:
            seen_ids.add(pid)
            algo_results.append(hit)
    
    # Fall back to general search if nothing found
    if not algo_results:
        print("  [TOOL] No algorithm content found, falling back to general search...")
        response = oai_client.embeddings.create(model=EMBED_MODEL, input=f"{winning_approach} reasoning training")
        query_vector = response.data[0].embedding
        algo_results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=5,
            with_payload=True
        ).points
    
    # Step 2: Extract algorithm excerpts
    algorithm_excerpts = []
    paper_citations = []
    
    for hit in algo_results[:5]:  # Limit to 5 best
        meta = hit.payload.get('metadata', {})
        content_type = hit.payload.get('content_type', 'abstract')
        text = hit.payload.get('text', '')
        title = meta.get('title', 'Unknown')
        
        algorithm_excerpts.append(f"[{content_type.upper()}] From '{title}':\n{text}")
        paper_citations.append(meta.get('paper_id', 'unknown'))
    
    print(f"  [TOOL] Found {len(algorithm_excerpts)} relevant excerpts from papers")
    for i, exc in enumerate(algorithm_excerpts):
        # Show first line of each excerpt for debugging
        first_line = exc.split('\n')[0]
        print(f"  [TOOL]   {i+1}. {first_line[:80]}")
    
    # Step 3: Generate code grounded in actual paper algorithms
    excerpts_text = "\n\n---\n\n".join(algorithm_excerpts) if algorithm_excerpts else "No specific algorithm content found."
    
    prompt = f"""You are implementing {spec["label"]} to improve reasoning in a small language model.

Below are excerpts from research papers about reasoning post-training (reinforcement learning and distillation).
Use ONLY the excerpts that are relevant to the request — ignore any excerpts about unrelated topics.

═══ RESEARCH PAPER EXCERPTS ═══
{excerpts_text[:4000]}

═══ CODE REQUEST ═══
{description}

═══ REQUIREMENTS ═══
{spec["requirements"]}

Return ONLY the Python code, no explanations before or after."""

    # Reasoning models would spend the budget thinking before the code and can
    # return no visible content at all; this is a pure code request, so thinking
    # is switched off (models without the switch ignore the kwarg).
    response = oai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048,
        temperature=0.3,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    
    code = response.choices[0].message.content or ""
    
    # Clean up: strip thinking tags if present (some models wrap in <think>...</think>)
    if '<think>' in code and '</think>' in code:
        think_end = code.index('</think>') + len('</think>')
        code = code[think_end:].strip()
    
    # Strip markdown code fences if the model wrapped the output
    if code.startswith('```python'):
        code = code[len('```python'):].strip()
    if code.startswith('```'):
        code = code[3:].strip()
    if code.endswith('```'):
        code = code[:-3].strip()
    
    # Store the context that was used
    extracted_contexts.append({
        "type": "code_generation",
        "description": description,
        "winning_approach": winning_approach,
        "paper_citations": list(dict.fromkeys(paper_citations)),  # deduplicate
        "algorithm_excerpts_count": len(algorithm_excerpts),
        "algorithm_excerpts": algorithm_excerpts,
        "excerpts_text": excerpts_text[:4000]
    })
    
    print(f"  [TOOL] Code generated ({len(code)} chars)")
    print(f"  [TOOL] Based on papers: {', '.join(list(dict.fromkeys(paper_citations))[:3])}")
    
    return json.dumps({
        "description": description,
        "winning_approach": winning_approach,
        "code": code,
        "paper_citations": list(dict.fromkeys(paper_citations)),
        "algorithm_excerpts_used": len(algorithm_excerpts)
    }, indent=2)


# Test tools
info("Testing search_papers with content_type filter...")
test_result = search_papers("reinforcement learning verifiable rewards reasoning", content_type="all")
success(f"search_papers: Found {len(json.loads(test_result))} papers")

# Show content type distribution in collection
info("Checking content types in Qdrant...")
for ct in ['abstract', 'algorithm', 'results', 'equation', 'setup', 'discussion']:
    try:
        ct_filter = Filter(must=[FieldCondition(key="content_type", match=MatchValue(value=ct))])
        count = qdrant.count(collection_name=COLLECTION_NAME, count_filter=ct_filter).count
        info(f"  - {ct}: {count} chunks")
    except:
        pass

success("Tools ready for debate!")

---
## 3. Configure the Agents

| Agent | Role | Position | Tool |
|---|---|---|---|
| **Pro_GRPO** | Advocate | RL on verifiable rewards | `search_papers` |
| **Pro_Distill** | Advocate | Distilling a reasoning teacher | `search_papers` |
| **Judge** | Arbiter | Scores both, implements the winner | `generate_code` |

### Prompts that stop rather than loop

Each advocate's system message is blunt about the workflow — *search once, then
argue* — and ends every turn with `ARGUMENT_COMPLETE`. Both details are load
bearing. Without the first, an agent keeps searching for better evidence and
never argues. Without the second, the proxy has no way to know the turn is over
and burns through its auto-reply budget.

### `enable_thinking: False`, and why it is required

The gateway serves a reasoning model, and its thinking is billed against
`max_tokens`. In an agent loop a turn that exhausts its budget mid-thought
returns **no visible content**, and AG2 rejects a message with neither content
nor a tool call — the run dies with an unhelpful error about ChatCompletion
messages.

Debate turns are short and evidence-driven, so thinking is switched off here.
Models without the switch ignore the parameter, so the setting is safe to leave
in place across backends.

### Judge scoring

Five criteria at **20% each**: quality, efficiency, simplicity, practicality,
robustness. See the header for why the weighting is flat.

The judge is also told, explicitly, not to write code itself but to call
`generate_code` — otherwise it writes an implementation from memory, ungrounded
in any paper, which defeats the point of having the index.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, register_function
from langfuse import Langfuse
import asyncio

# Initialize Langfuse client for observability
# Note: AG2/AutoGen doesn't have native Langfuse integration like LangChain does.
# We use Langfuse for manual tracing of key events (tool calls, debate rounds).
langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)
langfuse.auth_check()

success("Langfuse initialized for observability")

# AG2 LLM configuration — uses the thinkube LLM Gateway (OpenAI-compatible)
# The gateway routes requests to the appropriate backend (Ollama, vLLM, etc.)
llm_config = {
    "config_list": [{
        "model": CHAT_MODEL,
        "api_key": LLM_GATEWAY_TOKEN,
        "base_url": f"{LLM_GATEWAY_URL}/v1",
        "price": [0, 0],
        # Reasoning models (Qwen3 family) think before they answer, and the
        # thinking is billed against max_tokens. In an agent loop a turn that
        # runs out of budget mid-thought returns no visible content, which AG2
        # rejects. Debate turns are short and tool-driven, so thinking is
        # switched off here; models without the switch ignore the kwarg.
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    }],
    "temperature": 0.7,
    "timeout": 300,
    "max_tokens": 2048,
}

info(f"AG2 configured with model: {CHAT_MODEL}")
info(f"AG2 gateway: {LLM_GATEWAY_URL}/v1")

# Helper function for safe termination check
def check_termination(x, terms):
    """Safely check if any termination term is in the message content."""
    content = x.get("content") if x else None
    if content is None:
        return False
    return any(term in content for term in terms)

# ═══════════════════════════════════════════════════════════════════════════════
# ADVOCATE 1: Pro-GRPO
# ═══════════════════════════════════════════════════════════════════════════════
# CRITICAL: The prompt must FORCE the model to use search results immediately
# Without explicit instructions, the model keeps searching instead of arguing

pro_grpo = AssistantAgent(
    name="Pro_GRPO",
    system_message="""You advocate for GRPO — reinforcement learning on verifiable rewards — as the way to improve reasoning in a small LLM.

WORKFLOW (follow EXACTLY):
1. Call search_papers ONCE to find evidence for GRPO / RLVR
2. IMMEDIATELY after receiving results, write your argument using those results
3. DO NOT call search_papers again - use what you found

CRITICAL: After your FIRST search returns results, you MUST write your argument.
Do NOT keep searching for "better" results. Use what you have.

Your argument format:
EVIDENCE: [cite 2-3 papers from search results with their key findings]
ARGUMENT: [make your case for GRPO based on the evidence]
ARGUMENT_COMPLETE

You must end with ARGUMENT_COMPLETE after writing your argument.""",
    llm_config=llm_config,
)

# ═══════════════════════════════════════════════════════════════════════════════
# ADVOCATE 2: Pro-Distill
# ═══════════════════════════════════════════════════════════════════════════════

pro_distill = AssistantAgent(
    name="Pro_Distill",
    system_message="""You advocate for distilling a large reasoning teacher — SFT on the teacher's reasoning traces — over running RL on the small model itself.

WORKFLOW (follow EXACTLY):
1. Call search_papers ONCE to find evidence for reasoning distillation
2. IMMEDIATELY after receiving results, write your argument using those results
3. DO NOT call search_papers again - use what you found

CRITICAL: After your FIRST search returns results, you MUST write your argument.
Do NOT keep searching for "better" results. Use what you have.

Your argument format:
EVIDENCE: [cite 2-3 papers from search results with their key findings]
ARGUMENT: [make your case for distillation based on the evidence]
ARGUMENT_COMPLETE

You must end with ARGUMENT_COMPLETE after writing your argument.""",
    llm_config=llm_config,
)

# ═══════════════════════════════════════════════════════════════════════════════
# JUDGE (Holistic Evaluation)
# ═══════════════════════════════════════════════════════════════════════════════

judge = AssistantAgent(
    name="Judge",
    system_message="""You are an impartial judge evaluating a debate on how to improve reasoning in a small LLM: GRPO (RL on verifiable rewards) versus distilling a large reasoning teacher.

SCORING (0.0 to 1.0 for each criterion):
1. QUALITY (20%) - Task performance metrics from cited papers
2. EFFICIENCY (20%) - Memory usage, compute cost, training time
3. SIMPLICITY (20%) - Implementation complexity, hyperparameters
4. PRACTICALITY (20%) - Tooling support, ecosystem maturity
5. ROBUSTNESS (20%) - Performance variance, generalization

WORKFLOW (follow EXACTLY):
1. Present scores for BOTH sides in a table
2. Calculate TOTAL for each side
3. Declare WINNER = side with HIGHER total (mandatory) — name it GRPO or Distillation
4. Call generate_code tool - DO NOT write code yourself
5. After tool returns, say DEBATE_COMPLETE

CRITICAL RULES:
- WINNER must be the side with higher total score
- DO NOT write any code - the generate_code tool handles that
- DO NOT include code blocks in your response
- Just call the tool and wait for the result""",
    llm_config=llm_config,
)

# === User Proxy for tool execution ===
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,  # Increased to allow tool call + response
    is_termination_msg=lambda x: check_termination(x, ["ARGUMENT_COMPLETE", "DEBATE_COMPLETE"]),
    code_execution_config=False,
)

# Register tools - advocates get search_papers, judge gets generate_code
register_function(search_papers, caller=pro_grpo, executor=user_proxy,
                  name="search_papers", description="Search papers for evidence supporting GRPO / RLVR")
register_function(search_papers, caller=pro_distill, executor=user_proxy,
                  name="search_papers", description="Search papers for evidence supporting reasoning distillation")
register_function(generate_code, caller=judge, executor=user_proxy,
                  name="generate_code", description="Generate code implementing the winning approach")

success("Debate agents configured!")
info("  - Pro_GRPO (advocate for RL on verifiable rewards)")
info("  - Pro_Distill (advocate for distilling a reasoning teacher)")
info("  - Judge (holistic evaluation across 5 criteria)")

---
## 4. Run the Debate

**What**: Two rounds of argument, then adjudication.

**How it is organised**:

1. **Round 1 — openings, in parallel.** Both advocates run concurrently under
   `asyncio.gather`. They are independent, so this halves the wall clock.
2. **Round 2 — rebuttals, in parallel.** Each advocate is handed the *opponent's*
   opening and asked to counter it. This is the round that produces the sharpest
   evidence, because refuting a specific claim is a narrower search than arguing
   a position.
3. **Judgement.** The judge receives all four texts, scores them, declares a
   winner and calls `generate_code`.

**A detail worth reading in the code**: the judge's termination check accepts
both `ALL_DONE` and `DEBATE_COMPLETE`. Its system message specifies one marker
and the round's prompt specifies the other; accepting either is what stops the
judge repeating itself until it hits the auto-reply cap.

**Expect about four minutes**, most of it generation. At roughly 30 tokens per
second and four arguments of ~2,000 tokens each, that is arithmetic rather than
a problem — watch it stream and you can read along.

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in Jupyter

# Debate topic
DEBATE_TOPIC = {
    "name": "GRPO vs Distillation for Reasoning",
    "question": (
        "To improve reasoning in a small (under-10B) LLM on tasks with programmatically "
        "verifiable answers — given access to both a reward checker and a large reasoning "
        "teacher — should we post-train with GRPO (RL on verifiable rewards) or distill "
        "the teacher's reasoning traces (SFT)?"
    ),
}


async def run_advocate_round(advocate, prompt, name):
    """Run a single advocate's argument asynchronously."""
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    
    # Create a fresh proxy for this advocate
    advocate_proxy = UserProxyAgent(
        name="Moderator",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=10,  # Allow tool call + response
        is_termination_msg=lambda x: check_termination(x, ["ARGUMENT_COMPLETE"]),
        code_execution_config=False,
    )
    
    # Register the tool
    register_function(search_papers, caller=advocate, executor=advocate_proxy,
                      name="search_papers", description="Search papers for evidence")
    
    # Run the advocate
    result = advocate_proxy.initiate_chat(
        advocate,
        message=prompt,
        silent=False,
    )
    
    # Extract the argument - look for LAST message from the advocate agent
    # that contains substantive content (EVIDENCE, ARGUMENT, or REBUTTAL)
    argument = ""
    for msg in reversed(result.chat_history):
        # Skip tool call messages and moderator messages
        msg_name = msg.get("name", "")
        content = msg.get("content", "") or ""
        
        # Look for the advocate's actual argument (not tool calls)
        if msg_name == advocate.name and content:
            # Skip if it's just a tool call suggestion
            if "Suggested tool call" in content:
                continue
            # This should be the actual argument
            argument = content
            break
    
    # If we didn't find it, fall back to any message with argument markers
    if not argument:
        for msg in reversed(result.chat_history):
            content = msg.get("content", "") or ""
            if content and ("EVIDENCE" in content or "REBUTTAL" in content) and "Moderator" not in msg.get("name", ""):
                # Make sure it's not the original prompt
                if "Search for papers" not in content and "YOUR POSITION" not in content:
                    argument = content
                    break
    
    return argument


async def run_full_debate():
    """Run a full debate with opening arguments AND rebuttals."""
    
    print("\n" + "═"*70)
    print(f"  DEBATE: {DEBATE_TOPIC['name']}")
    print("═"*70)
    print(f"\nQuestion: {DEBATE_TOPIC['question']}\n")
    
    extracted_contexts.clear()
    start_time = time.time()
    
    # ═══════════════════════════════════════════════════════════════
    # ROUND 1: Opening Arguments (Parallel)
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  ROUND 1: OPENING ARGUMENTS")
    print("═"*70)
    
    # Simplified prompts that are more direct
    grpo_opening_prompt = f"""Topic: {DEBATE_TOPIC['question']}

You argue FOR GRPO. Search for papers about GRPO and RL with verifiable rewards, then write your argument.

Remember: ONE search, then write your argument with EVIDENCE and end with ARGUMENT_COMPLETE."""

    distill_opening_prompt = f"""Topic: {DEBATE_TOPIC['question']}

You argue FOR distillation. Search for papers about distilling reasoning from a teacher model, then write your argument.

Remember: ONE search, then write your argument with EVIDENCE and end with ARGUMENT_COMPLETE."""

    # Run both opening arguments in parallel
    grpo_task = asyncio.create_task(
        run_advocate_round(pro_grpo, grpo_opening_prompt, "🔵 PRO-GRPO: Opening Argument")
    )
    distill_task = asyncio.create_task(
        run_advocate_round(pro_distill, distill_opening_prompt, "🔴 PRO-DISTILLATION: Opening Argument")
    )
    
    grpo_opening, distill_opening = await asyncio.gather(grpo_task, distill_task)
    
    round1_time = time.time() - start_time
    print(f"\n  [Round 1 completed in {round1_time:.1f}s]")
    
    # Debug: Show what was captured
    print(f"\n  Pro-GRPO opening captured: {len(grpo_opening)} chars")
    print(f"  Pro-Distillation opening captured: {len(distill_opening)} chars")
    
    # ═══════════════════════════════════════════════════════════════
    # ROUND 2: Rebuttals (Parallel) - Each sees opponent's argument!
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  ROUND 2: REBUTTALS")
    print("═"*70)
    
    # Only do rebuttals if we got opening arguments
    grpo_rebuttal = ""
    distill_rebuttal = ""
    
    if grpo_opening and distill_opening:
        grpo_rebuttal_prompt = f"""Your opponent argued:
{distill_opening[:1500]}

Counter their points. Search for papers that challenge their claims, then write your rebuttal.

Remember: ONE search, then REBUTTAL with evidence, end with ARGUMENT_COMPLETE."""

        distill_rebuttal_prompt = f"""Your opponent argued:
{grpo_opening[:1500]}

Counter their points. Search for papers that challenge their claims, then write your rebuttal.

Remember: ONE search, then REBUTTAL with evidence, end with ARGUMENT_COMPLETE."""

        # Run both rebuttals in parallel
        grpo_rebuttal_task = asyncio.create_task(
            run_advocate_round(pro_grpo, grpo_rebuttal_prompt, "🔵 PRO-GRPO: Rebuttal")
        )
        distill_rebuttal_task = asyncio.create_task(
            run_advocate_round(pro_distill, distill_rebuttal_prompt, "🔴 PRO-DISTILLATION: Rebuttal")
        )
        
        grpo_rebuttal, distill_rebuttal = await asyncio.gather(grpo_rebuttal_task, distill_rebuttal_task)
    else:
        print("  [Skipping rebuttals - missing opening arguments]")
    
    round2_time = time.time() - start_time - round1_time
    print(f"\n  [Round 2 completed in {round2_time:.1f}s]")
    
    # Debug: Show what was captured
    print(f"\n  Pro-GRPO rebuttal captured: {len(grpo_rebuttal)} chars")
    print(f"  Pro-Distillation rebuttal captured: {len(distill_rebuttal)} chars")
    
    # ═══════════════════════════════════════════════════════════════
    # FINAL: Judge Evaluation (Holistic - 5 Criteria)
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  ⚖️ JUDGE: Holistic Evaluation (5 Criteria)")
    print("═"*70)
    
    # Build judge prompt with whatever arguments we have
    judge_sections = []
    if grpo_opening:
        judge_sections.append(f"PRO-GRPO OPENING:\n{grpo_opening}")
    if distill_opening:
        judge_sections.append(f"PRO-DISTILLATION OPENING:\n{distill_opening}")
    if grpo_rebuttal:
        judge_sections.append(f"PRO-GRPO REBUTTAL:\n{grpo_rebuttal}")
    if distill_rebuttal:
        judge_sections.append(f"PRO-DISTILLATION REBUTTAL:\n{distill_rebuttal}")
    
    if not judge_sections:
        judge_sections.append("No arguments were presented. Evaluate based on general knowledge of GRPO vs reasoning distillation.")
    
    # Simplified judge prompt - let model call tools naturally
    judge_prompt = f"""Debate topic: {DEBATE_TOPIC['question']}

{chr(10).join(judge_sections)}

YOUR TASK:
1. Evaluate both sides on: QUALITY, EFFICIENCY, SIMPLICITY, PRACTICALITY, ROBUSTNESS
2. Declare the WINNER (GRPO or Distillation)
3. Use generate_code to implement the winning approach
4. After you receive the code, write ALL_DONE"""

    # Custom termination for judge - only terminate after code is generated
    def judge_termination(x):
        content = x.get("content") if x else None
        if content is None:
            return False
        # Terminate once the judge signals it is done after code generation;
        # the judge's system message says DEBATE_COMPLETE, this prompt says
        # ALL_DONE — both mean the same thing here.
        return "ALL_DONE" in content or "DEBATE_COMPLETE" in content
    
    judge_proxy = UserProxyAgent(
        name="Court",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=10,
        is_termination_msg=judge_termination,
        code_execution_config=False,
    )
    
    register_function(generate_code, caller=judge, executor=judge_proxy,
                      name="generate_code", description="Generate code for winning approach")
    
    judge_result = judge_proxy.initiate_chat(
        judge,
        message=judge_prompt,
        silent=False,
    )
    
    total_time = time.time() - start_time
    
    # ═══════════════════════════════════════════════════════════════
    # RESULTS
    # ═══════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print("  DEBATE COMPLETE")
    print("═"*70)
    print(f"\n  Round 1 (Openings):  {round1_time:.1f}s")
    print(f"  Round 2 (Rebuttals): {round2_time:.1f}s")
    print(f"  Judge:               {total_time - round1_time - round2_time:.1f}s")
    print(f"  ─────────────────────────────")
    print(f"  Total:               {total_time:.1f}s")
    
    # Extract generated code - look for tool output
    generated_code = None
    for msg in judge_result.chat_history:
        content = msg.get("content", "")
        # Look for the actual code in tool output (contains "code":)
        if content and '"code":' in content:
            try:
                import json
                code_data = json.loads(content)
                if 'code' in code_data:
                    generated_code = code_data['code']
                    break
            except:
                pass
        # Also check for code blocks
        if content and "```python" in content and "def " in content:
            generated_code = content
    
    return {
        "topic": DEBATE_TOPIC["name"],
        "question": DEBATE_TOPIC["question"],
        "grpo_opening": grpo_opening,
        "distill_opening": distill_opening,
        "grpo_rebuttal": grpo_rebuttal,
        "distill_rebuttal": distill_rebuttal,
        "judge_history": judge_result.chat_history,
        "generated_code": generated_code,
        "total_time": total_time,
        "round1_time": round1_time,
        "round2_time": round2_time,
        "contexts": list(extracted_contexts)
    }


# Run the full debate!
debate_result = asyncio.get_event_loop().run_until_complete(run_full_debate())

# Flush traces
langfuse.flush()
success(f"Debate complete! Traces sent to Langfuse.")

---
## 5. Debate Results

The full transcript: both openings, both rebuttals, the judge's scoring table
and verdict, then the generated implementation and the paper excerpts it was
built from.

**Read the citations, not only the conclusion.** The measure of this pipeline is
whether the arguments rest on real findings from real papers — head-to-head
comparisons, reported failures, stated conditions — or on plausible generalities
the model could have produced without any index at all. Specific claims
attributed to named papers mean retrieval did its job.

In [ ]:
# Display the full debate with rebuttals

output = f"""
# Debate Results: {debate_result['topic']}

**Question:** {debate_result['question']}

**Duration:** {debate_result['total_time']:.1f} seconds (Round 1: {debate_result['round1_time']:.1f}s, Round 2: {debate_result['round2_time']:.1f}s)

---

# ROUND 1: Opening Arguments

## Pro-GRPO Opening

{debate_result['grpo_opening']}

---

## Pro-Distillation Opening

{debate_result['distill_opening']}

---

# ROUND 2: Rebuttals

## Pro-GRPO Rebuttal

{debate_result['grpo_rebuttal']}

---

## Pro-Distillation Rebuttal

{debate_result['distill_rebuttal']}

---

# Judge's Verdict

"""

# Extract judge's evaluation — find the longest substantive message from the Judge
# (skip tool call suggestions and short responses like "DEBATE_COMPLETE").
# The judge's turn that carries the tool call is stored without a name, so
# select by exclusion: anything that is not the Court proxy and not a tool result.
best_verdict = ""
for msg in debate_result['judge_history']:
    content = msg.get('content', '') or ''
    if msg.get('name') != 'Court' and msg.get('role') != 'tool' and content:
        # Skip tool call suggestions
        if 'Suggested tool call' in content:
            continue
        # Skip short termination markers
        clean = content.replace('DEBATE_COMPLETE', '').replace('ALL_DONE', '').strip()
        if len(clean) > len(best_verdict):
            best_verdict = clean

if best_verdict:
    output += best_verdict + "\n\n"
else:
    output += "*Judge verdict not captured — check cell 10 output for details*\n\n"

output += """
---

# Generated Code (Winning Approach)

"""

generated_code = debate_result.get('generated_code')
if generated_code and generated_code.strip():
    code = generated_code
    if not code.strip().startswith('```'):
        output += f"```python\n{code}\n```"
    else:
        output += code
else:
    output += "*Code generation timed out or returned empty — try with a larger model (9B/27B)*"

# Show code generation context (deduplicated — only show the last attempt)
code_gen_contexts = [ctx for ctx in debate_result.get('contexts', []) if ctx.get('type') == 'code_generation']
if code_gen_contexts:
    ctx = code_gen_contexts[-1]  # Only show the last attempt
    output += f"\n\n---\n\n# Code Generation Context\n\n"
    output += f"**Description:** {ctx.get('description', 'N/A')}\n\n"
    output += f"**Winning Approach:** {ctx.get('winning_approach', 'N/A')}\n\n"
    # Deduplicate paper citations
    unique_citations = list(dict.fromkeys(ctx.get('paper_citations', [])))
    output += f"**Paper Citations:** {', '.join(unique_citations)}\n\n"
    output += f"**Algorithm Excerpts Used:** {ctx.get('algorithm_excerpts_count', 0)}\n\n"

    algorithm_excerpts = ctx.get('algorithm_excerpts', [])
    if algorithm_excerpts:
        output += "## Algorithm Excerpts from Papers\n\n"
        output += "*These excerpts were used to ground the code generation:*\n\n"
        for i, excerpt in enumerate(algorithm_excerpts, 1):
            output += f"### Excerpt {i}\n\n"
            output += f"```\n{excerpt[:1500]}{'...' if len(excerpt) > 1500 else ''}\n```\n\n"

display(Markdown(output))

---
## 6. Debate Summary

Timings, argument lengths, the winning approach, and which papers grounded the
generated code.

Argument length is a rough health signal: a few hundred characters usually means
an agent hit an error or terminated early, while three to four thousand is a
developed argument with evidence.

In [ ]:
# Summary statistics
print("═"*60)
print("DEBATE SUMMARY")
print("═"*60)
print(f"\nTopic: {debate_result['topic']}")
print(f"\nTiming:")
print(f"  Round 1 (Openings):  {debate_result['round1_time']:.1f}s")
print(f"  Round 2 (Rebuttals): {debate_result['round2_time']:.1f}s")
print(f"  Judge:               {debate_result['total_time'] - debate_result['round1_time'] - debate_result['round2_time']:.1f}s")
print(f"  ─────────────────────────────")
print(f"  Total:               {debate_result['total_time']:.1f}s")

print(f"\nArgument lengths:")
print(f"  Pro_GRPO Opening:      {len(debate_result.get('grpo_opening', ''))} chars")
print(f"  Pro_Distill Opening:   {len(debate_result.get('distill_opening', ''))} chars")
print(f"  Pro_GRPO Rebuttal:     {len(debate_result.get('grpo_rebuttal', ''))} chars")
print(f"  Pro_Distill Rebuttal:  {len(debate_result.get('distill_rebuttal', ''))} chars")

generated_code = debate_result.get('generated_code')
code_generated = bool(generated_code and generated_code.strip())
print(f"\nCode Generated: {'Yes' if code_generated else 'No'}")

# Show code generation context (deduplicated — only last attempt)
code_gen_contexts = [ctx for ctx in debate_result.get('contexts', []) if ctx.get('type') == 'code_generation']
if code_gen_contexts:
    ctx = code_gen_contexts[-1]
    unique_citations = list(dict.fromkeys(ctx.get('paper_citations', [])))
    print(f"  - Winning approach: {ctx.get('winning_approach', 'N/A')}")
    print(f"  - Algorithm excerpts used: {ctx.get('algorithm_excerpts_count', 0)}")
    print(f"  - Paper citations: {', '.join(unique_citations[:5])}")

print(f"\nLangfuse traces: {LANGFUSE_HOST}")
print("═"*60)

---
## Try Other Questions

The pipeline is not specific to this debate. Three further questions are listed
below, each genuinely contested in the current literature.

Switching topic takes three steps, and skipping the third is the usual mistake:

1. Change `DEBATE_TOPIC` in the debate cell.
2. Adjust the advocates' system messages so each argues the new position.
3. **Re-run notebook 02 with matching queries and anchors.** The agents can only
   cite what is indexed — a debate on a topic the corpus does not cover produces
   confident, ungrounded prose, which is exactly the failure RAG exists to
   prevent.

In [ ]:
# Additional debate topics to try (all current reasoning post-training questions)
ADDITIONAL_TOPICS = [
    {
        "name": "Process vs Outcome Rewards",
        "question": "Do process reward models (scoring each reasoning step) beat outcome-only rewards for training reasoning models?",
        "description": "Debate step-level supervision against final-answer-only supervision."
    },
    {
        "name": "Test-Time Compute vs Bigger Model",
        "question": "Is spending inference tokens on longer thinking a better investment than training or serving a larger model?",
        "description": "Debate scaling test-time compute against scaling parameters."
    },
    {
        "name": "On-Policy vs Teacher-Trace Distillation",
        "question": "Should a student model learn from its own sampled attempts (on-policy distillation) or from the teacher's reasoning traces?",
        "description": "Debate where the training signal should come from."
    }
]

print("Additional debate topics available:")
print("="*60)
for i, topic in enumerate(ADDITIONAL_TOPICS, 1):
    print(f"\n{i}. {topic['name']}")
    print(f"   Q: {topic['question']}")
    print(f"   {topic['description']}")

print("\n" + "="*60)
print("\nTo run a new debate:")
print("1. Update DEBATE_TOPIC in the debate cell")
print("2. Update agent prompts if needed (the agents cell)")
print("3. Re-run notebook 02 with matching queries if you need different papers indexed")

---
## Next Steps

**Act on the verdict** — either outcome is runnable here:

- **GRPO wins** → the `zebra-grpo` example: RL on verifiable puzzle rewards with
  Unsloth. Needs a GPU-enabled notebook server.
- **Distillation wins** → collect reasoning traces from the chat model already
  served on your gateway, keep the ones a verifier confirms, and fine-tune a
  student with LoRA.

**Then question the verdict.** Re-run with a different `SEED`, or after adding
papers, and see whether it holds. A verdict that flips on reruns is telling you
the evidence is thin — which is itself a finding.

---

## What this notebook demonstrates

**Debate as a reasoning structure.** One model asked "which is better?" produces
a balanced-sounding summary. Two models made to argue opposite sides, each
reading the other, surface the strongest objection to each position — and the
rebuttal round consistently produces sharper citations than the openings.

**Retrieval decides argument quality.** The agents know only what
`search_papers` returns. Reserved slots for prose, evidence and anchors are not
tidiness: without them the debate is argued from benchmark tables, and the paper
that founded the field is never cited even though it sits in the index.

**Grounded generation.** The judge does not write code from memory. It retrieves
algorithm blocks for the winning approach and builds from them, with citations
back to the papers.

**Parallelism where it is free.** Both advocates run concurrently in each round;
they are independent, so the round costs one turn rather than two.

**Reasoning models need explicit handling.** Thinking is billed against the
token budget. In an agent loop that turns into empty messages and a dead run —
switched off here, deliberately, in three separate places.

---

*Papers sourced from arXiv. Thank you to arXiv for use of its open access
interoperability.*